> **Public-release version:** Private animal-level inputs and source-derived observations have been removed from the notebook. The original modelling and plotting cells are retained. See the repository README for data requirements.

# NB18 — SD-calibrated stacked population violins (PE and GI)

**Purpose.** Reproduce the proposed black-and-white population figure with horizontal maximum violin width proportional to the **between-animal sample SD of eight model-estimated animal-level posterior medians** in each component. This is a *custom stacked violin-style display*, not a kernel-density violin plot.

**Panel A (PE).** R, T1, Ts, where Ts = T2 − T1. Source: user-validated NB10 v1.2 posterior group and animal CSVs. PE target and non-target are time-matched at 6 h; later endpoints are not time-matched.

**Panel B (GI).** R, D, T1, Ts. Source: user-validated NB15 baseline hierarchical donor-model posterior group and animal CSVs. GI Days 10 and 14 are repeated observations on the same 16 animals; this figure does not estimate a longitudinal rate.

**Vertical size.** The source **group posterior medians** of each component are scaled within each bar to sum to 100%. The original unscaled medians and their credible intervals remain in the accompanying group-summary CSV. This normalization is only for graphical presentation, because separate component medians need not add to exactly 100%.

**Horizontal width.** Maximum half-width = 0.41 plot units × component between-animal SD ÷ global maximum SD. The global maximum SD is 47.25 percentage points for GI Day 10 target R. All eight experimental groups use the same scale. A smooth sine-profile merely provides the violin-shaped outline; *it is not an estimated density*.

**Interpretation.** Width represents variation between model-derived animal point estimates, not uncertainty in the group mean, a raw colony-count SD, a probability density or a 95% credible interval. Tiny components may be too small for internal labels; the figure caption defines all components. Experimental and modelling limitations remain.

**Run sequentially** with the aggregate data CSV supplied in `figures/`.


In [ ]:
#@title NB18.01 — Load group-level calibrated figure data
from pathlib import Path
import pandas as pd
WORK=Path.cwd()/'NB18_PE_GI_Calibrated_Widths'
WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('figures')/'Figure5_Calibrated_Width_Data.csv'
if not INPUT.is_file():
    INPUT=Path('..')/'figures'/'Figure5_Calibrated_Width_Data.csv'
if not INPUT.is_file():
    raise FileNotFoundError('Group-level Figure5_Calibrated_Width_Data.csv required. Run from repository root or notebooks/.')
DATA=WORK/'Figure5_Calibrated_Width_Data.csv'
DATA.write_bytes(INPUT.read_bytes())
frame=pd.read_csv(DATA)
assert len(frame)==28 and set(frame.study)=={'PE','GI'}
assert frame.groupby(['study','group']).n_animals.first().eq(8).all()
assert frame.groupby(['study','group']).height_fraction.sum().sub(1).abs().max()<1e-8
print('Verified aggregate Figure 5 data; saved:',DATA)


In [ ]:
#@title NB18.02 — Render SD-calibrated PE and GI stacked violins
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import matplotlib.patheffects as pe
from matplotlib import rcParams

DATA = WORK / 'Figure5_Calibrated_Width_Data.csv'
OUTPUT = WORK
df = pd.read_csv(DATA)

# One physical width scale across both panels.
GLOBAL_SD_MAX=float(df.between_animal_SD.max())
MAX_HALF_WIDTH=0.41             # in x-axis units
SHAPE_POWER=1.18                # smooth taper; constant shape for all groups/components
STYLES = {
    'R': dict(facecolor='white', edgecolor='black', hatch=None),
    'D': dict(facecolor='white', edgecolor='black', hatch='////'),
    'T1': dict(facecolor='white', edgecolor='black', hatch='xxxx'),
    'Ts': dict(facecolor='#242424', edgecolor='black', hatch=None),
}

rcParams.update({'font.family':'DejaVu Serif', 'font.size':11, 'axes.linewidth':0.9,
                 'hatch.linewidth':0.5, 'svg.fonttype':'none'})
fig, axs = plt.subplots(1,2,figsize=(15.5,6.35),sharey=True,constrained_layout=False)
fig.subplots_adjust(left=.061,right=.991,bottom=.20,top=.90,wspace=.15)
for ax,study,panel,components,xticks in zip(
    axs,
    ['PE','GI'],
    ['A','B'],
    [['R','T1','Ts'],['R','D','T1','Ts']],
    [ ['6 h\nTarget','6 h\nNon-target','Later\nTarget','Later\nNon-target'],
      ['Day 10\nTarget','Day 10\nNon-target','Day 14\nTarget','Day 14\nNon-target'] ]
):
    for idx in range(4):
        subset=df[(df.study==study)&(df['index']==idx)]
        y0=0.0
        for comp in components:
            row=subset[subset.component==comp].iloc[0]
            h=100*float(row.height_fraction)
            y1=y0+h
            sd=float(row.between_animal_SD)
            max_width=MAX_HALF_WIDTH*sd/GLOBAL_SD_MAX
            # Keep actual statistical width. A finite outline stroke is the only floor.
            if y1>y0+1e-8 and max_width>0:
                t=np.linspace(0,1,101)
                ys=y0+(y1-y0)*t
                widths=max_width*np.sin(np.pi*t)**SHAPE_POWER
                coords=np.column_stack((np.r_[idx-widths,(idx+widths)[::-1]],
                                        np.r_[ys,ys[::-1]]))
                patch=Polygon(coords,closed=True,lw=0.85,zorder=3,joinstyle='round',**STYLES[comp])
                ax.add_patch(patch)
                # Black border continuity on small components.
                if h>=5.5 and max_width>=0.067:
                    ax.text(idx,(y0+y1)/2,comp,ha='center',va='center',fontsize=10.5,
                            fontweight='bold',color='white' if comp=='Ts' else 'black',zorder=5)
            y0=y1
        assert abs(y0-100)<0.0001,(study,idx,y0)
    ax.set_xlim(-.53,3.53)
    ax.set_ylim(0,100)
    ax.set_xticks(range(4),xticks)
    ax.set_yticks([0,25,50,75,100])
    ax.tick_params(axis='both',direction='out',length=4,width=.8,pad=7)
    ax.yaxis.set_tick_params(labelleft=True)
    for side in ['top','right']:
        ax.spines[side].set_visible(False)
    ax.set_title(f'{panel}  {study} population composition',fontsize=13,fontweight='bold',loc='left',pad=16)
    for item in ax.get_xticklabels():item.set_fontsize(10)
axs[0].set_ylabel('Population fraction (%)',fontsize=12)
# Avoid decorative legend and bottom formula per user's request. Figure caption defines the components.
for typ in ['png','svg']:
    p=OUTPUT/f'Figure_5_PE_GI_SD_Calibrated_Stacked_Violins.{typ}'
    fig.savefig(p,dpi=400,bbox_inches='tight',facecolor='white')
    print(p, p.stat().st_size)
plt.close(fig)


In [ ]:
#@title NB18.03 — Display the numerical calibration table
frame=pd.read_csv(DATA)
view=frame[['study','group','component','median','q025','q975','between_animal_SD','height_fraction']].copy()
for col in ['median','q025','q975','between_animal_SD','height_fraction']:
    view[col]=100*view[col]
view=view.rename(columns={'median':'group_median_%','q025':'CrI_low_%','q975':'CrI_high_%',
                          'between_animal_SD':'between_animal_SD_pp','height_fraction':'plotted_height_%'})
from IPython.display import display
display(view.round(3))
print('Output directory:',WORK)
